# Hidden State Analysis (Dynamic NIAH v2, Colab)

This notebook is updated for the current Dynamic NIAH v2 workflow after the recent PRs:

- uses the active `scripts/gather_paul_graham_essays_v2.py`, `scripts/generate_dynamic_niah_v2.py`, `scripts/gen_responses.py`, and `scripts/analyze_hidden_states.py` entry points;
- writes experiments to timestamped run directories under `results/` with `figures/`, `tensors/`, `generate_data/`, `tables/`, `logs.txt`, and metadata files;
- supports config-driven prompt style, thinking mode, run naming, control-switch generation, and optional extra output copies;
- includes the newer hidden-state PCA trajectory figures produced by `analyze_hidden_states.py`.

Run the cells top to bottom in Colab. Edit only the configuration cell unless you need a custom workflow.


## 1. Mount Drive and choose the repo

Set `REPO_DIR` to the folder containing this repository on Drive. If you cloned the repo into `/content` instead, point `REPO_DIR` there.


In [2]:
from google.colab import drive
from pathlib import Path
import os
import sys

drive.mount('/content/drive')

# CHANGE THIS to your checked-out dataset-generation repository.
REPO_DIR = Path('/content/drive/MyDrive/Colab Notebooks/compression/dataset-generation-main-v6')

if not REPO_DIR.exists():
    raise FileNotFoundError(
        f'REPO_DIR does not exist: {REPO_DIR}\n'
        'Update REPO_DIR to your dataset-generation checkout before continuing.'
    )

os.chdir(REPO_DIR)
if str(REPO_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_DIR / 'src'))

print('Working directory:', Path.cwd())
print('Python path includes:', REPO_DIR / 'src')
# !git status --short


Mounted at /content/drive
Working directory: /content/drive/MyDrive/Colab Notebooks/compression/dataset-generation-main-v6
Python path includes: /content/drive/MyDrive/Colab Notebooks/compression/dataset-generation-main-v5/src


In [1]:
# Colab/runtime dependencies.
# Run this after mounting Drive and selecting REPO_DIR so subsequent imports can use the checkout.
# A later install cell can also use the repo's requirements.txt.
!pip -q install -U transformers accelerate datasets sentencepiece tqdm matplotlib


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 101.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 151.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 51.7 MB/s eta 0:00:00


In [3]:
# Install the repository-pinned requirements now that the working directory is the repo root.
# !pip -q install -U -r requirements.txt accelerate datasets sentencepiece tqdm


In [4]:
# Optional: pull the latest committed notebook/code before running experiments.
# Uncomment if this Colab checkout is connected to the repository remote.
# !git pull --ff-only


## 2. Experiment settings

The notebook computes one canonical `RUN_NAME` with `dataset_generation.run_utils.build_run_name(...)` and passes that name to every script. Running the cells sequentially therefore writes dataset generation, response generation, and hidden-state analysis into the same run folder under `RESULTS_ROOT`. Set `USER_RUN_NAME` only when you want to override the generated name with your own stable folder name.


In [5]:
from datetime import datetime
from pathlib import Path

#from dataset_generation.dynamic_niah_v2 import load_config_file
from dataset_generation.run_utils import build_run_name

MODEL_NAME = 'Qwen/Qwen3-8B'
CONFIG_PATH = 'configs/niah_dynamic.json'
# Runtime artifacts stay on the Colab VM, never in /content/drive.
RESULTS_ROOT = '/content/dataset_generation_results/colab_hidden_states'
# Only the final consolidated archive is moved to Drive.
DRIVE_ARCHIVE_DIR = Path('/content/drive/MyDrive/Colab Notebooks/compression/dataset-generation-archives')
USER_RUN_NAME = None  # e.g. 'qwen3_8b_argmax_easier_len1000_control0'

NUM_EXAMPLES = 20
TASK_TYPE = 'argmax'  # 'argmax' | 'count_avg'
TARGET_HAYSTACK_TOKENS = 1000
NUM_NEEDLES = 3
INSERTION_POSITIONS = [100, 200, 400]
CONTROL_SWITCH = [True, False, False]  # exactly one control needle is the default hidden-state comparison setup
PROMPT_STYLE = 'easier'  # 'easier' or 'vanilla'
LAYERS = [4, 8, 12, 16, 20, 24, 28]
HEADS = None  # None means all heads; otherwise use a list such as [0, 1, 7, 15]
MASSIVE_NORM_RATIO_THRESHOLD = 10.0
MASSIVE_TOP_K_PER_LAYER = 50
N_CRITICAL_EDGE_TOKENS = 10
N_AFTER_NEEDLE = 10
NEEDLE_SENSITIVE_TOP_M = 20
NEEDLE_SENSITIVE_EXPANSION = 5
SKIP_NEEDLE_SENSITIVE = False  # Keep False to generate tables/needle_sensitive_tokens.{json,txt}.
RUN_QK_OUTLIER_ANALYSIS = True  # Q/K capture and all-head analysis are expensive; set False to skip Section 11.
QK_CAPTURE_ATTN_IMPLEMENTATION = 'sdpa'  # Use 'flash_attention_2' only after installing a compatible flash-attn package.
QK_CAPTURE_MODEL_DTYPE = 'bf16'
QK_CAPTURE_SAVE_DTYPE = 'bf16'
SAVE_DATA = True
DELETE_LARGE_FILE_WHEN_DONE = True  # Deletes >200 MB .pt artifacts after Q/K analysis and downstream inspection cells.

# Compute one run name once so all later script calls reuse the same run folder.
#_cfg_payload_for_run_name = load_config_file(CONFIG_PATH)
#TASK_TYPE = _cfg_payload_for_run_name.get('task_type', 'argmax')
RUN_STARTED_AT = datetime.now()
RUN_NAME = USER_RUN_NAME or build_run_name(
    model_name=MODEL_NAME,
    params={
        'task': TASK_TYPE,
        'prompt': PROMPT_STYLE,
        'len': TARGET_HAYSTACK_TOKENS,
        'needles': NUM_NEEDLES,
    },
    start_time=RUN_STARTED_AT,
)
RUN_DIR = Path(RESULTS_ROOT) / RUN_NAME

# Keep Hugging Face model downloads local during runtime to avoid Drive sync storms.
HF_CACHE_DIR = Path('/content/huggingface_models')
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['TRANSFORMERS_CACHE'] = str(HF_CACHE_DIR)

print('Model:', MODEL_NAME)
print('Results root:', RESULTS_ROOT)
print('Run name:', RUN_NAME)
print('Run dir:', RUN_DIR)
print('HF cache:', HF_CACHE_DIR)
print('Drive archive dir:', DRIVE_ARCHIVE_DIR)
print('Drive will receive exactly one archive file when you run the archive/export cell near the end.')


Model: Qwen/Qwen3-8B
Results root: results/colab_hidden_states
Run name: run_20260530_030519_Qwen_Qwen3-8B_task-argmax_prompt-easier_len-1000_needles-3
Run dir: results/colab_hidden_states/run_20260530_030519_Qwen_Qwen3-8B_task-argmax_prompt-easier_len-1000_needles-3
HF cache: /content/drive/MyDrive/Colab Notebooks/huggingface_models


## 3. Validate imports and inspect the active config


In [6]:
import json
import torch
from dataclasses import asdict, replace
from pathlib import Path

from dataset_generation.dynamic_niah_v2 import DynamicNiahV2Config, load_config_file
from dataset_generation.hidden_state_analysis import (
    compare_hidden_states,
    compute_alignment_offset,
    compute_pca_projection_2d,
    project_hidden_states_with_pca,
)

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Total memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))

cfg_payload = load_config_file(CONFIG_PATH)
cfg = DynamicNiahV2Config(**cfg_payload)
cfg = replace(
    cfg,
    tokenizer_name=MODEL_NAME,
    task_type=TASK_TYPE,
    num_examples=NUM_EXAMPLES,
    target_haystack_tokens=TARGET_HAYSTACK_TOKENS,
    num_needles=NUM_NEEDLES,
    insertion_positions=tuple(INSERTION_POSITIONS),
    control_switch=CONTROL_SWITCH,
    prompt_style=PROMPT_STYLE,
    save_data=SAVE_DATA,
    results_root=RESULTS_ROOT,
    run_name=RUN_NAME,
    cache_dir=str(HF_CACHE_DIR),
)
print(json.dumps(asdict(cfg), indent=2, ensure_ascii=False))


CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
Total memory (GB): 39.49
{
  "task_type": "argmax",
  "tokenizer_name": "Qwen/Qwen3-8B",
  "num_examples": 20,
  "target_haystack_tokens": 1000,
  "num_needles": 3,
  "insertion_positions": [
    100,
    200,
    400
  ],
  "prompt_style": "easier",
  "thinking_mode": false,
  "output_dir": null,
  "data_save_path": null,
  "output_pred_jsonl": null,
  "output_metrics_json": null,
  "results_root": "results/colab_hidden_states",
  "run_dir": null,
  "run_name": "run_20260530_030519_Qwen_Qwen3-8B_task-argmax_prompt-easier_len-1000_needles-3",
  "global_random_seed": 42,
  "haystack_dir": "data/haystacks/paul_graham",
  "entities_path": "data/entities/cities.csv",
  "fact_templates_path": "data/templates/niah_fact_templates.txt",
  "haystack_seed": null,
  "needle_seeds": null,
  "control_switch": [
    true,
    false,
    false
  ],
  "save_data": true,
  "max_new_tokens": null,
  "temperature": 0.0,
  "trust_remote_code": true,
  "devic

In [7]:
# Fast local sanity checks for helper functions used by the analysis script.
a = torch.tensor([[1, 2, 3, 4, 5, 6]])
b = torch.tensor([[9, 9, 1, 2, 3, 4, 5, 6]])
offset = compute_alignment_offset(a, b, insertion_position=0, max_search_offset=4)
print('alignment offset:', offset)
assert offset == 2

H = torch.randn(3, 8, 5)
Hc = H.clone()
Hc[:, 4:, :] += 0.01
m = compare_hidden_states(H, Hc, insertion_position=4, offset=0, layer_indices=[0, 2])
print({k: tuple(v.shape) for k, v in m.items() if hasattr(v, 'shape')})
assert m['relative_norm_diff'].shape == (2, 8)
assert m['positions'].tolist() == list(range(8))
assert torch.isfinite(m['relative_norm_diff']).all()

H_shifted = torch.randn(3, 10, 5)
H_shifted[:, :4, :] = H[:, :4, :]
H_shifted[:, 6:, :] = H[:, 4:, :]
m_shifted = compare_hidden_states(H, H_shifted, insertion_position=4, offset=2, layer_indices=[0])
assert m_shifted['positions'].tolist() == list(range(8))
assert m_shifted['control_positions'].tolist() == [0, 1, 2, 3, 6, 7, 8, 9]
assert torch.allclose(m_shifted['relative_norm_diff'], torch.zeros(1, 8))

projection, mean = compute_pca_projection_2d(torch.randn(10, 5))
print('PCA projection shape:', tuple(projection.shape), 'mean shape:', tuple(mean.shape))
assert projection.shape == (5, 2)
projected = project_hidden_states_with_pca(
    H, H_shifted, 0, projection, mean, start_position=4, insertion_position=4, offset=2
)
assert projected['positions'].tolist() == [4, 5, 6, 7]
assert projected['control_positions'].tolist() == [6, 7, 8, 9]


[hidden-analysis] token-length mismatch/alignment  normal_len=6 control_len=8 insertion_position=0 offset=2
alignment offset: 2
{'relative_norm_diff': (2, 4), 'cosine_similarity': (2, 4), 'layers': (2,), 'insertion_position': (), 'offset': ()}
PCA projection shape: (5, 2) mean shape: (5,)


## 4. Prepare haystacks

The v2 gather script is the current Paul Graham corpus preparation path. It is safe to rerun; it refreshes/filters the haystack folder used by the generator.


In [ ]:
!PYTHONPATH=src python scripts/gather_paul_graham_essays_v2.py --out-dir data/haystacks/paul_graham


## 5. Optional: generate only the Dynamic NIAH v2 dataset

You can run this cell to inspect generated rows without loading a model for hidden states. The canonical output is under local `/content/...` via `RESULTS_ROOT/run_.../generate_data/`. Nothing is written to Google Drive until the archive/export cell runs.


In [ ]:
positions_args = ' '.join(str(p) for p in INSERTION_POSITIONS)
control_json = json.dumps(CONTROL_SWITCH).lower()
run_name_arg = f' --run-name {RUN_NAME}' if RUN_NAME else ''

cmd = (
    'PYTHONPATH=src python scripts/generate_dynamic_niah_v2.py '
    f'--config {CONFIG_PATH} '
    f'--task-type {TASK_TYPE} '
    f'--tokenizer \"{MODEL_NAME}\" '
    f'--num-examples {NUM_EXAMPLES} '
    f'--target-haystack-tokens {TARGET_HAYSTACK_TOKENS} '
    f'--num-needles {NUM_NEEDLES} '
    f'--positions {positions_args} '
    f'--prompt-style {PROMPT_STYLE} '
    f'--results-root {RESULTS_ROOT} '
    f"--control-switch-json '{control_json}'"
    f'{run_name_arg}'
)
print(cmd)
# Uncomment to run dataset generation only.
! $cmd


## 6. Optional: generate responses and metrics

`gen_responses.py` is the current end-to-end response generation workflow. It writes predictions to `tables/predictions.jsonl` and metrics to `tables/metrics.json` inside the run directory.

The dataset-size arguments (`target_haystack_tokens`, `num_needles`, and `insertion_positions`) are passed from the configuration cell below, so changing `NUM_NEEDLES`, `INSERTION_POSITIONS`, or `CONTROL_SWITCH` in one place keeps Section 6 aligned.


In [ ]:
positions_args = ' '.join(str(p) for p in INSERTION_POSITIONS)
control_args = ' '.join('true' if x else 'false' for x in CONTROL_SWITCH)
run_name_arg = f' --run-name {RUN_NAME}' if RUN_NAME else ''

cmd = (
    'PYTHONPATH=src python scripts/gen_responses.py '
    f'--config {CONFIG_PATH} '
    f'--model "{MODEL_NAME}" '
    f'--task-type {TASK_TYPE} '
    f'--num-examples {NUM_EXAMPLES} '
    f'--target-haystack-tokens {TARGET_HAYSTACK_TOKENS} '
    f'--num-needles {NUM_NEEDLES} '
    f'--positions {positions_args} '
    f'--prompt-style {PROMPT_STYLE} '
    f'--results-root {RESULTS_ROOT} '
    f'--control_switch {control_args}'
    f'{run_name_arg}'
)
print(cmd)
# Uncomment to run response generation.
! $cmd


## 7. Run hidden-state analysis

This is the main experiment cell. It generates a control-aware dataset, compares normal/control hidden states, and saves both measurement plots and PCA trajectory plots in the local `/content` run directory.


In [ ]:
layers_args = ' '.join(str(x) for x in LAYERS)
positions_args = ' '.join(str(p) for p in INSERTION_POSITIONS)
control_args = ' '.join('true' if x else 'false' for x in CONTROL_SWITCH)
run_name_arg = f' --run-name {RUN_NAME}' if RUN_NAME else ''
save_data_arg = 'true' if SAVE_DATA else 'false'
skip_needle_sensitive_arg = ' --skip-needle-sensitive' if SKIP_NEEDLE_SENSITIVE else ''

cmd = (
    'PYTHONPATH=src python scripts/analyze_hidden_states.py '
    f'--config {CONFIG_PATH} '
    f'--model \"{MODEL_NAME}\" '
    f'--task-type {TASK_TYPE} '
    f'--num-examples {NUM_EXAMPLES} '
    f'--target-haystack-tokens {TARGET_HAYSTACK_TOKENS} '
    f'--num-needles {NUM_NEEDLES} '
    f'--positions {positions_args} '
    f'--prompt-style {PROMPT_STYLE} '
    f'--layers {layers_args} '
    f'--control_switch {control_args} '
    f'--save_data {save_data_arg} '
    f'--needle-sensitive-top-m {NEEDLE_SENSITIVE_TOP_M} '
    f'--needle-sensitive-expansion {NEEDLE_SENSITIVE_EXPANSION} '
    f'--results-root {RESULTS_ROOT}'
    f'{run_name_arg}'
    f'{skip_needle_sensitive_arg}'
)
print(cmd)
! $cmd

if not SKIP_NEEDLE_SENSITIVE:
    expected_needle_sensitive_paths = [
        Path(RUN_DIR) / 'tables' / 'needle_sensitive_tokens.json',
        Path(RUN_DIR) / 'tables' / 'needle_sensitive_tokens.txt',
    ]
    missing_needle_sensitive_paths = [
        path for path in expected_needle_sensitive_paths if not path.exists()
    ]
    if missing_needle_sensitive_paths:
        raise FileNotFoundError(
            'Hidden-state analysis completed without the expected needle-sensitive outputs: '
            + ', '.join(str(path) for path in missing_needle_sensitive_paths)
        )
    for path in expected_needle_sensitive_paths:
        print(f'Verified needle-sensitive output: {path} ({path.stat().st_size} bytes)')


## 8. Inspect the configured run folder


In [ ]:
from pathlib import Path
import json

current_run = Path(RUN_DIR)
if not current_run.exists():
    raise FileNotFoundError(f'Configured run directory does not exist yet: {current_run}. Run an experiment cell first.')

# Keep the old variable name for the display/loading cells below.
latest_run = current_run
print('Configured run:', latest_run)
print('Run name:', RUN_NAME)
print('\nTop-level artifacts:')
for p in sorted(latest_run.iterdir()):
    print(' -', p.relative_to(latest_run), '(dir)' if p.is_dir() else f'({p.stat().st_size} bytes)')

metadata_path = latest_run / 'run_metadata.json'
if metadata_path.exists():
    print('\nrun_metadata.json:')
    print(metadata_path.read_text(encoding='utf-8')[:4000])

analysis_config_path = latest_run / 'analyze_hidden_states_config.json'
if analysis_config_path.exists():
    print('\nanalyze_hidden_states_config.json:')
    print(analysis_config_path.read_text(encoding='utf-8')[:4000])


In [ ]:
# Show generated data / metrics when present.
for candidate in [
    latest_run / 'generate_data' / 'dynamic_niah_v2.jsonl',
    latest_run / 'tables' / 'metrics.json',
    latest_run / 'tables' / 'predictions.jsonl',
    latest_run / 'tables' / 'needle_sensitive_tokens.json',
    latest_run / 'tables' / 'needle_sensitive_tokens.txt',
    latest_run / 'logs.txt',
]:
    print('\n' + '=' * 100)
    print(candidate)
    if candidate.exists():
        text = candidate.read_text(encoding='utf-8')
        print(text[:5000])
    else:
        print('missing')


## 9. Display saved figures


In [ ]:
from IPython.display import Image, display

figures = sorted((latest_run / 'figures').glob('*.png'))
print(f'Found {len(figures)} figure(s).')
for p in figures[:12]:
    print(p.name)
    display(Image(filename=str(p)))


## 10. Load tensors for custom follow-up analysis


In [ ]:
import torch

tensors = sorted((latest_run / 'tensors').glob('*.pt'))
print(f'Found {len(tensors)} tensor file(s).')
if tensors:
    sample = torch.load(tensors[0], map_location='cpu')
    print('Loaded:', tensors[0])
    for k, v in sample.items():
        if hasattr(v, 'shape'):
            print(k, tuple(v.shape), v.dtype)
        else:
            print(k, v)


## 11. Q/K-cache outlier analysis (uncontrolled prompts only)

This section reuses/generates Q/K caches for the all-needle uncontrolled prompts, computes received-attention sink statistics, detects massive hidden-state activations, and joins the two outlier views. The run-level settings are saved to `qk_outlier_analysis_config.json`, while per-example Q/K metadata is consolidated under `tensors/qk_cache/qk_cache_metadata.json`.

The default Q/K capture attention implementation is `sdpa`, because standard Colab runtimes usually do not include a compatible `flash-attn` build. If you install `flash-attn` yourself, change `QK_CAPTURE_ATTN_IMPLEMENTATION` in Section 2 to `flash_attention_2`.



In [ ]:
import json
import torch

from dataset_generation.qk_hook_attention.outlier_analysis import (
    QKOutlierAnalysisConfig,
    run_qk_outlier_analysis,
    save_qk_outlier_config,
)

qk_config = QKOutlierAnalysisConfig(
    model=MODEL_NAME,
    run_dir=str(RUN_DIR),
    layers=tuple(LAYERS),
    heads=None if HEADS is None else tuple(HEADS),
    massive_norm_ratio_threshold=MASSIVE_NORM_RATIO_THRESHOLD,
    massive_top_k_per_layer=MASSIVE_TOP_K_PER_LAYER,
    n_critical_edge_tokens=N_CRITICAL_EDGE_TOKENS,
    n_after_needle=N_AFTER_NEEDLE,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    capture_attn_implementation=QK_CAPTURE_ATTN_IMPLEMENTATION,
    capture_model_dtype=QK_CAPTURE_MODEL_DTYPE,
    capture_save_dtype=QK_CAPTURE_SAVE_DTYPE,
)
config_path = save_qk_outlier_config(RUN_DIR, qk_config)
print('Saved Q/K outlier config:', config_path)
print(json.dumps(qk_config.to_json_dict(), indent=2))

if RUN_QK_OUTLIER_ANALYSIS:
    qk_summary = run_qk_outlier_analysis(
        run_dir=RUN_DIR,
        layers=LAYERS,
        heads=HEADS,
        model=MODEL_NAME,
        repo_root=Path.cwd(),
        massive_norm_ratio_threshold=MASSIVE_NORM_RATIO_THRESHOLD,
        massive_top_k_per_layer=MASSIVE_TOP_K_PER_LAYER,
        n_critical_edge_tokens=N_CRITICAL_EDGE_TOKENS,
        n_after_needle=N_AFTER_NEEDLE,
        device='cuda' if torch.cuda.is_available() else 'cpu',
        capture_attn_implementation=QK_CAPTURE_ATTN_IMPLEMENTATION,
        capture_model_dtype=QK_CAPTURE_MODEL_DTYPE,
        capture_save_dtype=QK_CAPTURE_SAVE_DTYPE,
        figures_dir=qk_config.figures_dir,
        tables_dir=qk_config.tables_dir,
    )
    print(json.dumps(qk_summary, indent=2))
else:
    print('Set RUN_QK_OUTLIER_ANALYSIS = True in Section 2 to generate/reuse Q/K caches and run the analysis.')


## 12. Inspect Q/K outlier outputs


In [ ]:
from pathlib import Path

qk_candidates = [
    Path(RUN_DIR) / 'qk_outlier_analysis_config.json',
    Path(RUN_DIR) / 'tables' / 'qk_outlier_analysis_summary.json',
    Path(RUN_DIR) / 'tables' / 'massive_tokens_all.csv',
    Path(RUN_DIR) / 'tables' / 'massive_scalar_activations_all.csv',
    Path(RUN_DIR) / 'tables' / 'massive_activation_dim_counts.csv',
    Path(RUN_DIR) / 'tables' / 'attention_sinks_topk.csv',
    Path(RUN_DIR) / 'tables' / 'needle_attention_mass.csv',
    Path(RUN_DIR) / 'tables' / 'outlier_attention_join.csv',
    Path(RUN_DIR) / 'tables' / 'outlier_overlap_summary.csv',
    Path(RUN_DIR) / qk_config.figures_dir / 'inputs_0_qk_outliers.png',
    Path(RUN_DIR) / qk_config.tables_dir / 'inputs_0_qk_outliers.csv',
]
for candidate in qk_candidates:
    print('\n' + '=' * 100)
    print(candidate)
    if candidate.exists():
        text = candidate.read_text(encoding='utf-8')
        print(text[:5000])
    else:
        print('missing')


In [ ]:
# Optional: delete oversized .pt files after all analysis that needs them is complete.
# Q/K outlier analysis and needle-sensitive-token generation run before this cleanup cell,
# so the default True deletes large .pt artifacts only after all downstream analyses are complete.
from dataset_generation.hidden_state_analysis import prune_large_pt_files

if DELETE_LARGE_FILE_WHEN_DONE:
    deleted_or_flagged = prune_large_pt_files(Path(RUN_DIR) / 'tensors', delete=True)
    print(f'Deleted {len(deleted_or_flagged)} oversized .pt file(s).')
else:
    flagged = prune_large_pt_files(Path(RUN_DIR) / 'tensors', delete=False)
    print(f'Kept {len(flagged)} oversized .pt file(s). Set DELETE_LARGE_FILE_WHEN_DONE=True to delete them after analysis.')



## 12. Export one consolidated archive to Google Drive

Run this after generation/analysis is complete. It creates a `.zip` under `/content` first, then moves that single archive file to `DRIVE_ARCHIVE_DIR` so Google Drive never has to sync thousands of individual runtime artifacts.


In [ ]:
from pathlib import Path
import shutil

run_dir = Path(RUN_DIR)
if not run_dir.exists():
    raise FileNotFoundError(f'Configured run directory does not exist yet: {run_dir}. Run an experiment cell first.')
if not str(run_dir.resolve()).startswith('/content/') or str(run_dir.resolve()).startswith('/content/drive/'):
    raise RuntimeError(f'RUN_DIR must be local to /content and outside /content/drive before export: {run_dir}')

local_archive_root = Path('/content/dataset_generation_archives')
local_archive_root.mkdir(parents=True, exist_ok=True)
DRIVE_ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

local_base = local_archive_root / run_dir.name
local_zip = Path(shutil.make_archive(str(local_base), 'zip', root_dir=run_dir.parent, base_dir=run_dir.name))
drive_zip = DRIVE_ARCHIVE_DIR / local_zip.name
if drive_zip.exists():
    drive_zip.unlink()
shutil.move(str(local_zip), drive_zip)
print('Moved one consolidated archive to Drive:', drive_zip)
print('Runtime directory remains local:', run_dir)


## 13. Needle-sensitive tokens

Section 7 now requests `scripts/analyze_hidden_states.py` to generate `tables/needle_sensitive_tokens.json` and `tables/needle_sensitive_tokens.txt` for every run unless `SKIP_NEEDLE_SENSITIVE=True`. This cell verifies both files exist and prints the lowest-cosine non-needle tokens by layer.


In [ ]:
import json
from collections import Counter
from pathlib import Path

needle_sensitive_json_path = Path(RUN_DIR) / 'tables' / 'needle_sensitive_tokens.json'
needle_sensitive_txt_path = Path(RUN_DIR) / 'tables' / 'needle_sensitive_tokens.txt'
missing_paths = [p for p in [needle_sensitive_json_path, needle_sensitive_txt_path] if not p.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Needle-sensitive token outputs are missing. Re-run Section 7 with '
        'SKIP_NEEDLE_SENSITIVE=False before cleanup/archive. Missing: '
        + ', '.join(str(p) for p in missing_paths)
    )

print('Needle-sensitive JSON:', needle_sensitive_json_path)
print('Needle-sensitive text:', needle_sensitive_txt_path)
needle_sensitive = json.loads(needle_sensitive_json_path.read_text(encoding='utf-8'))
for layer, rows in sorted(needle_sensitive.get('by_layer', {}).items(), key=lambda item: int(item[0])):
    print(f'Layer {layer}')
    for row in rows[:NEEDLE_SENSITIVE_TOP_M]:
        print(
            f"  example={row['sample_idx']} t={row['position']} "
            f"control_t={row['control_position']} C[t]={row['cosine_similarity']:.6f} "
            f"token={row['token']!r}"
        )
    token_counts = Counter(row['token'] for row in rows)
    repeated = token_counts.most_common(10)
    print('  repeated tokens:', repeated)
    print()
# Optional: release the Colab VM when the run is complete.
from google.colab import runtime
runtime.unassign()
